# IRS Example: GL->F1065 Multimember Example

Refer to `IRSGuide/LLC-IRS1065Example.md`


In [1]:
# Load GL

In [2]:
#Form 1065 Worksheet using LLC General ledger accounts
# ════════════════════════════════════════════════════════════════════════════

# Init Services
from __future__ import annotations
import os
from pathlib import Path
from IPython.display import display, Markdown

import math
from dataclasses import dataclass, field
from typing import Dict, Optional
import pandas as pd

# --------- Link to General Ledger/IRS objects - All accounts
# LLC General Ledger
from ledger.LLC import LLC

# ---- Link to LLC ledgers & GL (glDict)
top = Path.cwd().parents[2]
llcName = [f for f in os.listdir(top) if 'llcProfile' in f][0].replace('.json','').replace('llcProfile_','')
llc = LLC(llcName,debug=False, top=top)
glDict  = round(llc.acctsDF(),2).to_dict()

# ---- Save entity information
eDict = llc.entity
acctDIR = llc.acctDir() #os.path.join(llc.TOP, llc.dirAccounting, str(llc.yr))
yeDIR = llc.acctDir(dirName='ye')



# Save entity information
eDict = llc.entity

glDict


llcAssets: FAIL load: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/Accts/llcAssets_WBGroupLLC.json, Expecting ':' delimiter: line 30 column 26 (char 1010)


AttributeError: 'DataFrame' object has no attribute 'dt'

In [ ]:

aList = llc.assets().load()
[aDict for aDict in aList if aDict['category'] == 'Asset.Equity.Cash' and str(llc.yr) in aDict['dt']][0]
aList[0]['amt']

In [ ]:
print("Ledger rows", len(llc.bk.df))
llc.bk.df.tail(5)

In [3]:
# llcTax/llc_Tax_EIN ---- Entity/EIN
oNm = lambda d : ', '.join(d['nm'])
oDict = {d['oID']:d for d in llc.owners()}
aDict = {d['oID']:d for d in llc.assets().load()}

sect = lambda cls : cls(llc).sec()
sept = lambda cls : cls(llc).sep()
toDF = lambda secList : pd.DataFrame(secList)
wsDF = lambda secList : toDF(secList)[['descWS','v']].set_index('descWS')

class llcTax(object):
    # Core class for all sections
    def __init__(self, llc, **kwargs):
        self.oID = self.__class__.__name__
        self.llc = llc
        # load GenLedger : glDict
        self.glDict = round(self.llc.acctsDF(),2).to_dict()

        # Income
        self.rev = self.glDict['Acct.Cash.Income']
        self.incInt = self.glDict['Acct.Interest.Income']+ self.glDict['Acct.Cash.Misc']
        self.gross = self.rev + self.incInt

        # Deductable expenses - active Rental
        self.expCash = glDict['Acct.Cash.Expense']
        self.expUtil =  glDict['Acct.Cash.Util']
        
        # Future expense deductable when active
        self.expNA = 0

        # Total Expenses
        self.exp = self.expCash + self.expUtil - self.expNA
        
    def sec(self):
        # return EntityName only
        llc = self.llc
        
        # --- Build header 
        # Entity
        secList = [dict(v=llc.entity['entity_name'], glNm='entity_name', fmKey=['P1_Hdr_3'], descWS='Entity')]
        return secList
        
    def sep(self):
        # return a separator line
        secList = [dict(v=self.oID, glNm=None, fmKey = [], descWS=f"{'-'*30}")]
        return secList + self.sec()


class llcTax_EIN(llcTax):
    # LLC EIN

    def sec(self):
        secList = []
        
        # EIN
        secList.append(dict(v=llc.entity['ein'], glNm='EIN', fmKey = ['P1_D'], descWS='EIN'))
        
        return secList



In [4]:
# llcTax_Members - member % ownership
class llcTax_Members(llcTax):
    # Member distribution

    def sec(self):
        secList = []
        llc=self.llc
        
        # ---- Ownership 
        for i,oID in enumerate(oDict):
            d = oDict[oID]
            desc = f"Member: oNm(d) [status:{d['status']} / oID:{oID} ]"
            secList.append(dict(v=f"{100*d['pct']:0.1f}%", glNm=f"own_{i+1}", fmKey=['SchK1.P1_A'], descWS=desc))
        return secList

In [45]:
llcTax(llc).glDict['Balance']

6719.18

In [5]:
# llcTax_Inc - All Incomes
class llcTax_Inc(llcTax):
    '''
    Income Section of 1065
    '''

    def sec(self):
        secList = []
        llc=self.llc

        
        # All Income
        net = self.gross + self.exp
        # --- Revenue
        secList.append(dict(v=f"${self.rev:0.2f}", glNm='fyRev', fmKey = ['P1_1a', 'P1_8'], 
                            descWS='Line 1a,1c: Gross receipts/sales'))
        incOther = self.glDict['Acct.Interest.Income'] + self.glDict['Acct.Cash.Misc']
        secList.append(dict(v=f"${incOther:0.2f}", glNm='incOther', fmKey = ['P1_7'], 
                            descWS='Line 7: Other Income (Int + Misc.Cash)'))
        # ---- income Sum
        secList.append(dict(v=f"${self.gross:0.2f}", glNm='fyNet', fmKey = ['P1_23'], descWS='Line 8: Total Income income (gros)'))
        return secList
        


In [6]:
# llcTax_Exp - Expenses and deductions
class llcTax_Exp(llcTax):
    '''
    Income Section of 1065
    '''
    
    def sec(self):
        secList = []
        llc=self.llc
        
        # --- Expenses
        secList.append(dict(v=f"${self.exp:0.2f}", glNm='fyExp', fmKey = ['P1_21'], 
                            descWS='Line 21: Other Expenses'))
        # --- Balance - Net
        secList.append(dict(v=f"${self.exp:0.2f}", glNm='taxNet', fmKey = ['P1_22'], descWS='Line 22: Total Deducations (exp)'))
        return secList

In [7]:
# llcTax_Bal
class llcTax_Bal(llcTax):
    # 1095 - Net (Ordinary business income(loss) & tax due (if any)
    
    def sec(self):
        secList = []
        llc=self.llc
        
        # --- Balance - Net
        net = self.gross + self.exp
        secList.append(dict(v=f"${net:0.2f}", glNm='taxNet', fmKey = ['P1_23'], descWS='Line 23: Ordinary bus income/loss (net)'))

        # --- Balance - Net
        secList.append(dict(v=f"${0:0.2f}", glNm='taxDue', fmKey = ['P1_28','P1_31'], descWS='Line 28: Total Balance Due (Tax)'))

        # --- Sign checkbox
        secList.append(dict(v=f"Ck", glNm='SelfEmpEarn', fmKey = ['P1_Sign_1'], descWS='Line Sign Checkbox: No to Paid Preparer'))

        return secList
        

In [8]:
# llcTax_Dist - Member Distribution

class llcTax_Dist(llcTax):
    # Worksheet Header

    def sec(self):
        secList = []
        llc=self.llc

        # --- Financials 
        gross = self.gross
        exp = self.exp # is negative
        net = gross + exp
        
        # ---- Ownership 
        for i,oID in enumerate(oDict):
            d = oDict[oID]
            desc = f"SchK1.P3_1: Member: oNm(d), {llc.yr} Cash Distribution"
            secList.append(dict(v=f"{net*d['pct']:0.1f}%", glNm=f"dist_{i+1}", fmKey=['SchK1.P3_1'], descWS=desc))
        return secList

In [9]:
# llcTax_SchB - Page 2-4
class llcTax_SchB(llcTax):
    '''
    Computes taxs for Page 1 of 1065
    '''

    def sec(self):
        secList = []
        llc=self.llc

        secList.append(dict(v="Ck", glNm='B_1b', fmKey = ['B_1_2'], descWS='Box 1.b: Type of Entity'))
        secList.append(dict(v="Ck", glNm='B_2b', fmKey = ['B_2_3'], descWS='Box 2.b: >50% Member Interest, Need Sch B-1'))
        diDict = {k:v for k,v in self.llc.F1065.items() if 'B_PRDI_' in k}
        for i,glNm in enumerate(diDict):
            
            # Split glNm into parts
            l = glNm.split('_')
            
            # Get prefix B_PRDI
            p = '_'.join(l[0:-1])
            desc=f'SchB Designated Rep: {l[-1]}'
            
            secList.append(dict(v=diDict[glNm], glNm=glNm, fmKey = [f"{p}_{i+1}"], descWS = desc))

        return secList
        
        

'''
## IRS Guide: Purchase 805HighMesa
- Sch L : report (ledger) the LLC's assets, liabilities, and capital at the start and end of the year. 
- The property purchase is a capital investment, not an immediate deduction.
- partners provide funds to the LLC to buy property, this is a capital contribution
- Assets:
    - Acct.Cash are Asset, ie.  Balance Sheet (Schedule L) account - liquid money the LLC has in its bank.
    - Acct.Asset is the Capital Account (Equity): each partner's ownership stake and "right" to those assets.
    - FIXME : change Acct.Asset to Acct.Capital

 #### Lines 2a/2b must match the total of Item L on all Sch K1 (sent to partners)
    - is the Sum of K-1s
    - must match the total of Item L (Partner’s Capital Account Analysis) on all Schedule K-1s issued to partners, 
    - specifically looking at the net income/loss allocated.
    - Separate Types: If a partnership has both general and limited partners, or if an LLC has members who are not active in daily management (limited) vs. those who are (general), you must split the profit/loss accordingly.
    - Treatment of Losses: If the partnership had a loss, the amounts are entered as negative numbers. 

#### Assets (Lines 1-14): e.g., Cash, Accounts Receivable.
#### Liabilities (Lines 15-20): e.g., Accounts Payable, Mortgages.
#### Partners' Capital Accounts (Line 21): This must match the ending balance on Schedule M-2. 

#### House Purchase
    - Property (Fixed Asset): Debit $220,662.25. :: This is cost basis.
        - Sale Price ($220,000) 
        - Closing Cost ($662.25) (title insurance/recording fees)
            - Tax Note: You must eventually split the basis
            - Land (non-depreciable) and 
            - Building (depreciable).
        - HOA Dues Debit $235.34 (Expense)
        - Owner Expense Costs (Recurring/prorated, eg. HOA 
        - typically recorded as an expense in the period they occur.
        - Earnest Money/Down Payment (Asset/Equity): Credit $5,300.00
        - If you already paid this before closing, you credit the Earnest Money/Escrow asset 
             account you used when the initial payment left your bank.
        - If you paid it out of pocket personally and are now contributing it to the LLC, 
            - credit Owner’s Equity/Capital Contribution.
        - If you paid cash from the business bank account, credit the Bank Account. 
#### Journal Entry
````
    General Ledger                                 Debit         Credit
    -------------------------------------------------------------------------
    Acct.Cash.Investment: Sale Price(Fixed Asset)  $214,700.00
    Acct.Cash.Investment: Sale Price(Earnest)        $5,300.00
    Acct.Cash.Expense: Title Co. Closing Costs         $662.25
    Acct.Cash.Expense: HOA Dues                        $235.34
    Acct.Cash.Credit: Prop Tax Payable                             -$1,660.64
    Acct.Cash.Investment: Earnest Money                            -$5,300.00
    Acct.Cash.Purchase: Cash at closing                          -$213,936.95
    -------------------------------------------------------------------------
    Closing Total Cash (Buyer/Seller)              $220,897.59	 -$220,897.59

    Equity Ledger
    -------------------------------------------------------------------------
    Acct.Equity.Tangible.InService                -$213,936.95
    Acct.Equity.Fixed                                             $220,000.00
    Acct.Equity.Fixed.Basis                                           $897.35
    Acct.Equity.Cash                                               -$6,960.64
    -------------------------------------------------------------------------
    Total                                          $220,897.59    $220,897.59

    Asset-Liability Ledger


   

Basis :: SalePrice + Title Cost - PropTaxPayable
````


- Schedule L (Balance Sheet): Report the asset on page 5.
- Line 9a: Investment real estate or "Other investments".
- Line 10a/11: Buildings and other depreciable assets at cost.
- Line 9b: Land (if applicable), which is not depreciable.
- Form 4562: Use this form to calculate and report depreciation for the building portion once it is "placed in service". 
'''

In [16]:
impList = [dict(nm = 'plumbing', dt='2025.04', amt=525.00+340.00),
           dict(nm='Foundaion', dt='2025.04', amt=1850.00),
           dict(nm='Electrical', dt='2025.04', amt=1046.52),
           dict(nm='Septic', dt='2025.04', amt=734.54),
           dict(nm='Paint,Flooring,Misc', dt='2023.12', amt=10500.00)
          ]
iDF = pd.DataFrame(impList)
iDF.to_dict(orient='records')


[{'nm': 'plumbing', 'dt': '2025.04', 'amt': 865.0},
 {'nm': 'Foundaion', 'dt': '2025.04', 'amt': 1850.0},
 {'nm': 'Electrical', 'dt': '2025.04', 'amt': 1046.52},
 {'nm': 'Septic', 'dt': '2025.04', 'amt': 734.54},
 {'nm': 'Paint,Flooring,Misc', 'dt': '2023.12', 'amt': 10500.0}]

In [ ]:
class llcTax_SchL(llcTax):

    def _begCash(self):
        aList = self.llc.assets().load()
        [aDict for aDict in aList if aDict['category'] == 'Asset.Equity.Cash' and str(llc.yr) in aDict['dt']][0]
        return aList[0]['amt']

    def _analysis(self:

        secList = []

        net = self.gross+self.exp
        
        aSum = 0 # active members
        pSum = 0 # passive members
        for oDict in self.llc.owners():
            amt = oDict['pct'] * net
            if oDict['memType'] = 'active' : 
                aSum += amt
            else: 
                pSum += amt

        # Analysis of Net Income x Members
        secList.append(dict(v=net, glNm='B_1b', fmKey = ['L_1'], descWS='Line 1: Analysis: Net Income(loss)'))
        secList.append(dict(v=aSum, glNm='B_1b', fmKey = ['L_ANI_1'], descWS='Line 1: Analysis: General x Active'))
        secList.append(dict(v=pSum, glNm='B_1b', fmKey = ['L_ANI_8'], descWS='Line 1: Analysis: Limited x Passive'))

        return secList

    def _assets(self):
        secList = []
        llc=self.llc

        # Line 1: Cash books Beg, End
        begCash = self.llc.assets()._begCash()
        endCash = self.glDict['Balance']
        secList.append(dict(v=begCash, glNm='Lbeg', fmKey = ['L_1_1'], descWS='Line 1: SchL: Beg Cash'))
        secList.append(dict(v=endCash, glNm='Lend', fmKey = ['L_1_3'], descWS='Line 1: SchL: End Cash'))

        # Line 3: Inventory - assets under termination - to be sold

        # Tangible, In serivce Assets -> Line 10a
        alist = [aDict for aDict in  if 'Acct.Tangible' in aDict['category']]])
        
        aAmtBeg = 0. # Depletable/InService asseet
        aAmtEnd = 0
        oAmtBeg = 0 # Not in Service
        oAmtEnd = 0
        for aDict in llc.assets().load():
            cat = aDict['category']
            if 'Acct.Tangible' not in cat: continue

            # Get asset amount
            amt = aDict['amt']

            # asset new in this year
            aNew = aDict['dt'][0:4] != str(self.llc.yr)
            
            if cat == 'Acct.Tangible.InService':
                aAmtEnd += amt
                if not aNew : aAmtBeg += amt

            else: # Other
                aAmtEnd += amt
                if not aNew : aAmtBeg += amt
            
        
        if aAmtEnd != 0.0 :
            secList.append(dict(v=aAmtBeg, glNm='L_T_beg', fmKey = ['L_10a_0'], descWS='Line 10a: SchL: Beg Depletable, Tangible.InService'))
            secList.append(dict(v=aAmtEnd, glNm='L_T_end', fmKey = ['L_10a_2'], descWS='Line 10a: SchL: End Depletable, Tangible.InService'))

        if oAmtEnd != 0.0: 
            secList.append(dict(v=oAmtBeg, glNm='L_O_beg', fmKey = ['L_13_1'], descWS='Line 13: SchL: Beg Other Assets'))
            secList.append(dict(v=oAmtEnd, glNm='L_O_end', fmKey = ['L_13_3'], descWS='Line 13: SchL: End Other Assets'))

        secList.append(dict(v=aAmtBeg+oAmtBeg, glNm='L_all_beg', fmKey = ['L_14_1'], descWS='Line 14: SchL: Total Asset'))
        secList.append(dict(v=aAmtEnd+oAmtEnd, glNm='L_all_beg', fmKey = ['L_14_3'], descWS='Line 10a: SchL: Total Assets'))

        return secList

    def _liability(self):
        '''
        - Capitals
            - House
                - Line 9b (Land):  portion of the purchase price allocated to land (not depreciable).
                - Line 10a (Buildings): Record the portion of the purchase price allocated to the structure.
                - Line 10b (Accumulated Depreciation): record the current year's (while in service) depreciation here.
            - RV Purchase: Line 14 (Other Assets): 
                - total cash spent so far on the RV as "Construction in Progress" 
                - do not put this on the "Buildings and other depreciable assets" line 
                - because it is not ready for use.
        - record loans
            - RV loan on line 17 (short term)

        
        '''
        secList = []
        llc=self.llc

        

    def sec(self):
        return self._analysis() + self._assets() + self._liabilities()

        

In [48]:
s = 112800
l = 63720
l/(s+l), s/(s+l)

(0.3609789259007478, 0.6390210740992522)

In [12]:
class llcTax_M1(llcTax):
    '''
    '''

    def sec(self):
        secList = []
        llc=self.llc
        secList.append(dict(v='', glNm='glTBD', fmKey = ['F'], descWS = 'Line : desc'))

        return secList
 

In [13]:
class llcTax_M2(llcTax):
    '''
    On page 5, the total of all partner contributions is recorded on Line 2a (Cash) or Line 2b (Property). 
    Schedule M-2, Analysis of Partners' Capital Accounts: 
    - Line 2: total cash contributions for the year are reported here (Line 2).
    
    '''

    def sec(self):
        secList = []
        llc=self.llc
        secList.append(dict(v='', glNm='glTBD', fmKey = ['F'], descWS = 'Line : desc'))

        return secList
 

In [14]:
# llcTax_F4562 - Property Purchase
class llcTax_F4562(llcTax):
    '''
    Record Property Purchase
    '''

    def sec(self):
        secList = []
        llc=self.llc
        secList.append(dict(v='', glNm='glTBD', fmKey = ['F'], descWS = 'Line : desc'))

        return secList
 

In [15]:
# llcTax_SchK - Page 5
class llcTax_SchK(llcTax):
    '''
    Schedule K-1, Item L: Report the contribution in the "Analysis of partner's capital account" section.
    Capital contributed during the year: Enter the cash or the adjusted basis of property contributed.

    Part II, Item L: The ending capital account balance should reflect the new cash contribution. 

    Line 1: non rental income
    Line 2: rental income

    '''

    def sec(self):
        secList = []
        llc=self.llc

        net = self.gross + self.exp
        secList.append(dict(v=net, glNm='K_net', fmKey = ['K_1'], descWS = 'Line 1: Ord. Bus. Inome Line 23'))
        secList.append(dict(v=self.rev, glNm='K_incRent', fmKey = ['K_2'], descWS = 'Line 2: Net rental income'))
        secList.append(dict(v=self.glDict['Acct.Cash.Misc'], glNm='KincO', fmKey = ['K_11'], descWS = 'Line 11: Other income (Int+Misc)'))
        
        return secList

In [16]:
class llcTax_SchK1(llcTax):
    '''
    Schedule K-1 (Form 1065), Part II, Item J: The cash contribution increases the partner's capital account. 
           It should be reported under "Capital contributed during the year".
    
    '''

    def sec(self):
        secList = []
        llc=self.llc
        secList.append(dict(v='', glNm='glTBD', fmKey = ['F'], descWS = 'Line : desc'))

        return secList
 

In [17]:
class llcTax_F8825(llcTax):
    '''
    Rental Real Estate Income and Expenses of LLC Partnership
    '''

    def sec(self):
        secList = []
        llc=self.llc
        secList.append(dict(v='', glNm='glTBD', fmKey = ['F'], descWS = 'Line : desc'))

        return secList
 

In [18]:
class llcTax_(llcTax):
    '''
    '''

    def sec(self):
        secList = []
        llc=self.llc
        secList.append(dict(v='', glNm='glTBD', fmKey = ['F'], descWS = 'Line : desc'))

        return secList
 

In [19]:
class llcTax_(llcTax):
    '''
    '''

    def sec(self):
        secList = []
        llc=self.llc
        secList.append(dict(v='', glNm='glTBD', fmKey = ['F'], descWS = 'Line : desc'))

        return secList
 

## Worksheet Summary

In [20]:
# Display Worksheet Summar

                
display(Markdown("#### 1. Scenario Overview"))
display(wsDF(sect(llcTax) + sect(llcTax_EIN)
             + sect(llcTax_Members)
             +sect(llcTax_Inc)
             +sect(llcTax_Exp)
             +sect(llcTax_Bal)
             + sect(llcTax_Dist)
            ))
            

display(Markdown("#### 2. Worksheet: Form 1065 - Dist. Summary"))
display(wsDF(sect(llcTax) + sect(llcTax_EIN)
             +sect(llcTax_Inc)
             +sect(llcTax_Exp)
             +sect(llcTax_Bal)
             + sect(llcTax_Dist)
            ))

display(Markdown("#### 3. Form 1065 - Return of Partnership Income"))
display(wsDF(sect(llcTax) + sect(llcTax_EIN)
             +sect(llcTax_Inc)
             +sect(llcTax_Exp)
             +sect(llcTax_Bal)
             +sept(llcTax_SchB)
             +sept(llcTax_SchK)
            ))



#### 1. Scenario Overview

,v
descWS,
Entity,"W&B Group, LLC"
EIN,39-3842347
Member: oNm(d) [status:Manager / oID:o20250801-1 ],96.0%
Member: oNm(d) [status:non_active member / oID:o20250801-2 ],2.0%
Member: oNm(d) [status:non-active member / oID:o20250801-3 ],2.0%
"Line 1a,1c: Gross receipts/sales",$4000.53
Line 7: Other Income (Int + Misc.Cash),$429.47
Line 8: Total Income income (gros),$4430.00
Line 21: Other Expenses,$-2823.87


#### 2. Worksheet: Form 1065 - Dist. Summary

,v
descWS,
Entity,"W&B Group, LLC"
EIN,39-3842347
"Line 1a,1c: Gross receipts/sales",$4000.53
Line 7: Other Income (Int + Misc.Cash),$429.47
Line 8: Total Income income (gros),$4430.00
Line 21: Other Expenses,$-2823.87
Line 22: Total Deducations (exp),$-2823.87
Line 23: Ordinary bus income/loss (net),$1606.13
Line 28: Total Balance Due (Tax),$0.00


#### 3. Form 1065 - Return of Partnership Income

,v
descWS,
Entity,"W&B Group, LLC"
EIN,39-3842347
"Line 1a,1c: Gross receipts/sales",$4000.53
Line 7: Other Income (Int + Misc.Cash),$429.47
Line 8: Total Income income (gros),$4430.00
Line 21: Other Expenses,$-2823.87
Line 22: Total Deducations (exp),$-2823.87
Line 23: Ordinary bus income/loss (net),$1606.13
Line 28: Total Balance Due (Tax),$0.00
